# Análise Exploratória de Vendas de Videogames

**Disciplina:** Estatística | **Integrantes:**                                                                                                                                  Arthur Sindeaux de Araújo Nogueira
Guilherme Tolentino Leitão de Melo
Rafael Coutinho Lima
Lucas Pinto Ribeiro Beno
Pedro Coutinho da Silva
Arthur Oliveira Furieri
João Henrique Bastos
João Eduardo Azevedo de Andrade

## Problema principal

Analisar as vendas de títulos de videogame para entender como o desempenho comercial se distribui entre gêneros, plataformas e regiões.

## Perguntas de investigação e hipóteses

**Pergunta 1.** Como se distribui o desempenho comercial dos títulos, e quão concentrado está o mercado em poucos lançamentos?
*Hipótese 1:* a distribuição é fortemente assimétrica à direita, com média bastante superior à mediana e o terceiro quartil ainda em patamar baixo. Se confirmado, o jogo mediano vende uma fração do que a média sugere, e a média é uma métrica enganosa para planejar um lançamento.

**Pergunta 2.** O desempenho de um título em um mercado se repete nos demais, ou cada região tem dinâmica própria?
*Hipótese 2:* as receitas regionais não se movem juntas de forma uniforme. Esperamos divergência entre Pearson e Spearman, porque Pearson é dominado pelos poucos títulos gigantes enquanto Spearman descreve o jogo típico. Se as duas medidas apontarem direções diferentes, a previsibilidade entre mercados só existe na faixa dos grandes sucessos.

**Pergunta 3.** Gêneros e plataformas com mais lançamentos são também os de melhor desempenho típico?
*Hipótese 3:* volume de lançamentos não acompanha desempenho mediano. Esperamos que categorias mais exploradas apresentem mediana comprimida por saturação.

## Fonte dos dados

Dataset `vgsales.csv` (Video Game Sales, Kaggle), com estimativas de vendas em milhões de unidades. Cada linha representa **um jogo em uma plataforma específica**: o mesmo título lançado em dois consoles ocupa duas linhas.

# 1 Config do Ambiente

In [1]:
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np


## 2. Carga e reconhecimento inicial dos dados

In [2]:
df = pd.read_csv("data/vgsales.csv")

print("Linhas e colunas:", df.shape)
df.head()
df.info()
df.describe()

Linhas e colunas: (16598, 11)
<class 'pandas.DataFrame'>
RangeIndex: 16598 entries, 0 to 16597
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Rank          16598 non-null  int64  
 1   Name          16598 non-null  str    
 2   Platform      16598 non-null  str    
 3   Year          16327 non-null  float64
 4   Genre         16598 non-null  str    
 5   Publisher     16540 non-null  str    
 6   NA_Sales      16598 non-null  float64
 7   EU_Sales      16598 non-null  float64
 8   JP_Sales      16598 non-null  float64
 9   Other_Sales   16598 non-null  float64
 10  Global_Sales  16598 non-null  float64
dtypes: float64(6), int64(1), str(4)
memory usage: 1.4 MB


,Rank,Year,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
count,16598.000000,16327.000000,16598.000000,16598.000000,16598.000000,16598.000000,16598.000000
mean,8300.605254,2006.406443,0.264667,0.146652,0.077782,0.048063,0.537441
std,4791.853933,5.828981,0.816683,0.505351,0.309291,0.188588,1.555028
min,1.000000,1980.000000,0.000000,0.000000,0.000000,0.000000,0.010000
25%,4151.250000,2003.000000,0.000000,0.000000,0.000000,0.000000,0.060000
50%,8300.500000,2007.000000,0.080000,0.020000,0.000000,0.010000,0.170000
75%,12449.750000,2010.000000,0.240000,0.110000,0.040000,0.040000,0.470000
max,16600.000000,2020.000000,41.490000,29.020000,10.220000,10.570000,82.740000


**Leitura inicial.** A base tem 16.598 registros e 11 colunas. Já aparecem dois problemas a tratar: `Year` foi lida como decimal, o que é sintoma de valores ausentes, e as colunas de venda têm média muito distante do máximo, primeiro indício da assimetria que a Pergunta 1 investiga.

## 3. Pré-processamento


In [3]:
nulos = pd.DataFrame({
    "ausentes": df.isnull().sum(),
    "percentual": (df.isnull().sum() / len(df) * 100).round(2)
})
nulos[nulos["ausentes"] > 0]

,ausentes,percentual
Year,271,1.63
Publisher,58,0.35
